# Add Mapped SMILES Columns

Add `Component1-Mapped` and `Component2-Mapped` columns to `training-set-renamed.csv` by converting plain SMILES to atom-mapped SMILES.

In [2]:
import pandas as pd
from openff.toolkit import Molecule

In [3]:
CSV_PATH = "../training-set-renamed.csv"

df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

(1155, 18)


,Id,Temperature (K),Pressure (kPa),Phase,N Components,Component 1,Role 1,Mole Fraction 1,Exact Amount 1,Component 2,Role 2,Mole Fraction 2,Exact Amount 2,Density Value (g / ml),Density Uncertainty (g / ml),EnthalpyOfMixing Value (kJ / mol),EnthalpyOfMixing Uncertainty (kJ / mol),Source
0,dens_3881511669765346,298.15,101.325,Liquid,2,C1COCCN1,Solvent,0.2980,NaN,O,Solvent,0.7020,NaN,1.0309,0.00255,NaN,NaN,10.1021/je020191g
1,dhmix_4129982627140506,298.20,101.300,Liquid,2,CCOC(=O)CC(=O)OCC,Solvent,0.2515,NaN,CO,Solvent,0.7485,NaN,NaN,NaN,1.0830,0.0115,10.1021/je9003806
2,dhmix_8746820869913363,298.15,100.000,Liquid,2,CC(=O)CC(C)=O,Solvent,0.2004,NaN,O,Solvent,0.7996,NaN,NaN,NaN,0.1562,0.0021,10.1016/j.fluid.2007.12.008
3,dhmix_9109633210018884,298.15,101.000,Liquid,2,CCCOC=O,Solvent,0.5035,NaN,ClCCCl,Solvent,0.4965,NaN,NaN,NaN,-0.2540,0.0030,10.1016/j.jct.2008.10.007
4,dhmix_14477488793854648,298.15,100.000,Liquid,2,CCCCC(=O)OC,Solvent,0.5055,NaN,COC(C)=O,Solvent,0.4945,NaN,NaN,NaN,0.2380,0.0025,10.1021/acs.jced.5b00813


In [4]:
def smiles_to_mapped_smiles(smiles: str) -> str | None:
    """
    Return a mapped SMILES string using the OpenFF toolkit canonical atom ordering.

    Atom-map numbers are assigned to all atoms (including hydrogens) via
    Molecule.to_smiles(mapped=True), which is consistent with how OpenFF
    internally indexes atoms for parameter assignment.
    """
    if not isinstance(smiles, str) or not smiles.strip():
        return None
    try:
        mol = Molecule.from_smiles(smiles, allow_undefined_stereo=True)
        return mol.to_smiles(mapped=True)
    except Exception:
        return None

In [5]:
df["Component1-Mapped"] = df["Component 1"].apply(smiles_to_mapped_smiles)
df["Component2-Mapped"] = df["Component 2"].apply(smiles_to_mapped_smiles)

print(f"Null Component1-Mapped: {df['Component1-Mapped'].isna().sum()}")
print(f"Null Component2-Mapped: {df['Component2-Mapped'].isna().sum()}")
df[["Component 1", "Component1-Mapped", "Component 2", "Component2-Mapped"]].head()

Null Component1-Mapped: 0
Null Component2-Mapped: 0


,Component 1,Component1-Mapped,Component 2,Component2-Mapped
0,C1COCCN1,[H:7][C:1]1([C:2]([O:3][C:4]([C:5]([N:6]1[H:15...,O,[H:2][O:1][H:3]
1,CCOC(=O)CC(=O)OCC,[H:12][C:1]([H:13])([H:14])[C:2]([H:15])([H:16...,CO,[H:3][C:1]([H:4])([H:5])[O:2][H:6]
2,CC(=O)CC(C)=O,[H:8][C:1]([H:9])([H:10])[C:2](=[O:3])[C:4]([H...,O,[H:2][O:1][H:3]
3,CCCOC=O,[H:14][C:5](=[O:6])[O:4][C:3]([H:12])([H:13])[...,ClCCCl,[H:5][C:2]([H:6])([C:3]([H:7])([H:8])[Cl:4])[C...
4,CCCCC(=O)OC,[H:9][C:1]([H:10])([H:11])[C:2]([H:12])([H:13]...,COC(C)=O,[H:9][C:4]([H:10])([H:11])[C:3](=[O:5])[O:2][C...


In [6]:
df.to_csv("../training-set-renamed-with-mapped-smiles.csv", index=False)
print(f"Saved {len(df)} rows with columns: {list(df.columns)}")

Saved 1155 rows with columns: ['Id', 'Temperature (K)', 'Pressure (kPa)', 'Phase', 'N Components', 'Component 1', 'Role 1', 'Mole Fraction 1', 'Exact Amount 1', 'Component 2', 'Role 2', 'Mole Fraction 2', 'Exact Amount 2', 'Density Value (g / ml)', 'Density Uncertainty (g / ml)', 'EnthalpyOfMixing Value (kJ / mol)', 'EnthalpyOfMixing Uncertainty (kJ / mol)', 'Source', 'Component1-Mapped', 'Component2-Mapped']
